# Занятие 4. Динамический контент: Playwright и no-code — демонстрационный блокнот

**Курс:** Автоматизированный сбор данных из интернета (код + no-code)
**Формат:** Демонстрация + база для лабораторной работы

Все примеры выполняются на **тренировочных площадках**, предназначенных для обучения скрапингу (`quotes.toscrape.com`, `scrapingcourse.com`, `books.toscrape.com`). Реальные коммерческие сайты на занятии не используются — это принципиальная установка курса.

### Содержание + замечания по блокноту

| Блок |На что обратить внимание |
|---|---|
| 0. Установка и запуск браузера | `playwright install`, headless, контексты |
| 1. Диагностика: почему BeautifulSoup пуст | сравнение «до JS» и «после JS» |
| 2. Данные в `<script>` — браузер не нужен | извлечение JSON регуляркой + `json.loads` |
| 3. Playwright: локаторы и извлечение | приоритет локаторов, `eval_on_selector_all` |
| 4. Ожидания | `sleep` vs автоожидание vs `expect_response` |
| 5. Бесконечная прокрутка |цикл с предохранителями |
| 6. Перехват XHR → уходим на HTTP |самый дешёвый путь, замер выигрыша |
| 7. Логин и `storage_state` | переиспользование сессии |
| 8. Экономика и no-code | блокировка ресурсов|


## 0. Установка и запуск

В аудитории браузеры должны быть установлены заранее: `pip install playwright` и `python -m playwright install chromium` (загрузка ~120 МБ). На занятии работаем в headless-режиме; `headless=False` показываем один раз, если позволяет окружение.

> **Sync или async API.** В Jupyter уже работает цикл событий asyncio, поэтому в блокноте используется **асинхронный** API Playwright с `await` (синхронный API внутри работающего цикла выдаёт ошибку). В обычных `.py`-скриптах студентам проще начинать с синхронного API: `with sync_playwright() as pw:` — код тот же, но без `await`. Это полезно проговорить, иначе студенты столкнутся с ошибкой при переносе кода.

Терминология: **browser** (процесс) → **context** (изолированная сессия: свои cookies и storage, дешёвая) → **page** (вкладка). Для параллельного сбора создают несколько контекстов в одном браузере, а не несколько браузеров.

1. Под Windows локально на ПК блокнот (.ipnyb) с PlayWright ни sync, ни async, ни отображение браузера, да и вообще с PlayWright не запускается. Блокнот можно запустить в Colab запустить удалось, правда без отображения браузера. Ниже ячейка - установка Playwright и браузера в среду Colab.
2. Согласно документации PlayWright он работает .py. Поэтому, если необходимо изучить функционал этого блокнота на ПК то → конвертируйте блокнот в обычный скрипт .py, тогда работу скрипта можно будет посмотреть (при режиме `headless=False`)  в запускающемся окне браузера. Паузы между блоками можно создать с помощью например,  `await page.pause()` или ожидания Enter (`await  ainput("...")`) и т.п.

3. ### ⚠️ Важное примечание: Управление памятью в Google Colab

Мы запускаем Playwright в скрытом режиме (`headless=True`). Браузер Chromium не виден на экране, но работает как **реальный фоновый процесс** на серверах Google.

* **В чем опасность?** Если в коде парсинга произойдет ошибка (например, не найдет селектор), выполнение ячейки прервется. Но сам процесс Chromium останется висеть в оперативной памяти Colab.
* **Что произойдет?** Если запустить код с ошибкой 5–10 раз подряд, «забытые» процессы забьют всю выделенную RAM. В результате Google Colab аварийно перезапустит всю сессию (`Crash`), и вы потеряете все накопленные в переменных данные.

* **Если ваш код падает** , не забывайте вручную выполнять последнюю ячейку с `await browser.close()` или используйте конструкцию `try...finally` для критических участков кода.


In [ ]:
!pip install playwright nest_asyncio
!playwright install --with-deps chromium

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:9 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,162 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,688 kB]
Fetched 3,981 kB in 2s (1,816 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading pack

In [ ]:
import asyncio, json, re, time, os
import requests
from bs4 import BeautifulSoup
import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

JS_URL     = "https://quotes.toscrape.com/js/"
SCROLL_URL = "https://quotes.toscrape.com/scroll"
LOGIN_URL  = "https://quotes.toscrape.com/login"
STATIC_URL = "https://quotes.toscrape.com/"

pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=True)   # headless=False — для показа в аудитории
context = await browser.new_context(
    user_agent="course-crawler/1.0 (educational use)",
    viewport={"width": 1280, "height": 900},
    locale="ru-RU",
)
context.set_default_timeout(15_000)              # 15 с на любое действие
page = await context.new_page()
print("Playwright запущен, версия браузера:", browser.version)

Playwright запущен, версия браузера: 153.0.8010.12


## 1. Диагностика: почему статический парсер видит пустую страницу

Сравниваем два среза одной и той же страницы: то, что отдал сервер (`requests`), и то, что стало после исполнения JS (Playwright). Это ключевой навык занятия — без него студенты начинают «лечить» проблему подбором селекторов.

In [ ]:
html_server = requests.get(JS_URL, timeout=15,
                           headers={"User-Agent": "course-crawler/1.0"}).text
quotes_server = BeautifulSoup(html_server, "html.parser").select(".quote")

await page.goto(JS_URL, wait_until="domcontentloaded")
await page.wait_for_selector(".quote")                # ждём результат работы JS
html_browser = await page.content()
quotes_browser = BeautifulSoup(html_browser, "html.parser").select(".quote")

print(f"requests      : {len(html_server):>6} байт HTML, найдено .quote = {len(quotes_server)}")
print(f"Playwright    : {len(html_browser):>6} байт HTML, найдено .quote = {len(quotes_browser)}")
print(f"Разница размера: {len(html_browser) - len(html_server):+} байт — это результат работы JS")

print("\nЕсть ли слово 'quote' в ответе сервера:", "quote" in html_server)
print("Есть ли текст первой цитаты в ответе сервера:",
      "The world as we have created it" in html_server)

requests      :   5806 байт HTML, найдено .quote = 0
Playwright    :   8940 байт HTML, найдено .quote = 10
Разница размера: +3134 байт — это результат работы JS

Есть ли слово 'quote' в ответе сервера: True
Есть ли текст первой цитаты в ответе сервера: True


### Контрольный опыт: статическая версия того же сайта

Тот же контент по адресу без `/js/` отдаётся сервером готовым. Полезно показать рядом: проблема не в сайте и не в библиотеке, а в способе формирования DOM.

In [ ]:
static_html = requests.get(STATIC_URL, timeout=15).text
print("Статическая версия: .quote =", len(BeautifulSoup(static_html, "html.parser").select(".quote")))
print("Вывод: один и тот же контент может требовать разных инструментов —")
print("решение принимается по результату диагностики, а не по привычке.")

Статическая версия: .quote = 10
Вывод: один и тот же контент может требовать разных инструментов —
решение принимается по результату диагностики, а не по привычке.


## 2. Данные лежат в `<script>` — браузер не нужен

Частый и самый выгодный случай: JS не запрашивает данные с сервера, а получает их встроенными в разметку (`var data = [...]`, `__NEXT_DATA__`, `application/ld+json`). Тогда достаточно обычного HTTP-запроса и разбора JSON — экономия на порядок.

In [ ]:
m = re.search(r"var data\s*=\s*(\[.*?\]);", html_server, re.S)
print("JSON-блок найден в исходном HTML:", bool(m))

t0 = time.perf_counter()
data = json.loads(m.group(1))
http_time = time.perf_counter() - t0

df_script = pd.json_normalize(data, sep="_")
df_script["tags"] = df_script["tags"].apply(lambda v: ", ".join(v) if isinstance(v, list) else v)
print(f"Записей получено: {len(df_script)} | разбор занял {http_time*1000:.1f} мс")
display(df_script[["author_name", "tags", "text"]].head(5))

JSON-блок найден в исходном HTML: True
Записей получено: 10 | разбор занял 0.1 мс


,author_name,tags,text
0,Albert Einstein,"change, deep-thoughts, thinking, world",“The world as we have created it is a process ...
1,J.K. Rowling,"abilities, choices","“It is our choices, Harry, that show what we t..."
2,Albert Einstein,"inspirational, life, live, miracle, miracles",“There are only two ways to live your life. On...
3,Jane Austen,"aliteracy, books, classic, humor","“The person, be it gentleman or lady, who has ..."
4,Marilyn Monroe,"be-yourself, inspirational","“Imperfection is beauty, madness is genius and..."


⚠️Фиксируем правило приоритетов:

1. Данные есть в HTML (`<script>`, JSON-LD, `__NEXT_DATA__`) → `requests` + `json.loads`.
2. Данные приходят отдельным XHR → `requests` к этому эндпоинту (занятие 3).
3. Данные рождаются только в браузере (вычисляются JS, требуют взаимодействия) → Playwright.

## 3. Playwright: локаторы и извлечение данных

Показываем приоритет локаторов и два способа извлечения: поэлементный перебор (читаемо, медленнее) и один вызов JS в странице (быстро, одна поездка в браузер).

In [ ]:
await page.goto(JS_URL, wait_until="domcontentloaded")

# --- ленивый локатор: вычисляется в момент обращения
quotes = page.locator(".quote")
print("Элементов на странице:", await quotes.count())

first = quotes.first
print("Автор первой цитаты  :", await first.locator(".author").inner_text())
print("Текст (обрезан)      :", (await first.locator(".text").inner_text())[:60], "…")

# --- локаторы по семантике: устойчивы к смене классов
print("\nЕсть ссылка 'Next'    :", await page.get_by_role("link", name="Next").count() > 0)
print("Тег 'love' на странице:", await page.get_by_role("link", name="love").count() > 0)

# --- фильтрация по тексту
love = page.locator(".quote").filter(has_text="love")
print("Цитат со словом 'love':", await love.count())

Элементов на странице: 10
Автор первой цитаты  : Albert Einstein
Текст (обрезан)      : “The world as we have created it is a process of our thinkin …

Есть ссылка 'Next'    : True
Тег 'love' на странице: False
Цитат со словом 'love': 1


In [ ]:
# Массовое извлечение одним вызовом в контексте страницы — быстрее, чем N обращений
t0 = time.perf_counter()
records = await page.eval_on_selector_all(".quote", """
els => els.map(el => ({
    text:   el.querySelector('.text')?.innerText?.trim() ?? null,
    author: el.querySelector('.author')?.innerText?.trim() ?? null,
    tags:   Array.from(el.querySelectorAll('.tag')).map(t => t.innerText)
}))
""")
bulk = time.perf_counter() - t0

t0 = time.perf_counter()
slow = [{"text": await q.locator(".text").inner_text(),
         "author": await q.locator(".author").inner_text()} for q in await quotes.all()]
per_item = time.perf_counter() - t0

print(f"Один вызов JS   : {len(records)} записей за {bulk*1000:.0f} мс")
print(f"Поэлементно     : {len(slow)} записей за {per_item*1000:.0f} мс")
print(f"Разница         : ×{per_item/max(bulk, 1e-9):.1f}")
display(pd.DataFrame(records).head(3))

Один вызов JS   : 10 записей за 37 мс
Поэлементно     : 10 записей за 143 мс
Разница         : ×3.9


,text,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"


### Приоритет локаторов (переписать студентам в конспект)

| Приоритет | Пример | Почему |
|---|---|---|
| 1 | `get_by_test_id("price")` | атрибут создан для автоматизации |
| 2 | `get_by_role("button", name="Загрузить ещё")` | семантика, не оформление |
| 3 | `get_by_text`, `get_by_label`, `get_by_placeholder` | видимый пользователю контракт |
| 4 | `[data-product-id]`, `[itemprop="price"]` | стабильные атрибуты данных |
| 5 | `.product-card .price` | классы компонента: терпимо |
| ✗ | `//div[3]/div[2]/span` | ломается от любой вставки блока |
| ✗ | `.css-1x7f2kq`, наборы Tailwind | меняются при каждой сборке фронтенда |

## 4. Ожидания: три уровня надёжности

Сравниваем на живой странице: фиксированная пауза, ожидание селектора (автоожидание) и ожидание конкретного сетевого ответа. Последнее — эталон, потому что мы ждём именно нужные данные.

In [ ]:
# --- Уровень 1: фиксированная пауза (антипаттерн)
p1 = await context.new_page()
t0 = time.perf_counter()
await p1.goto(JS_URL, wait_until="commit")
await asyncio.sleep(3)                           # «на всякий случай»
n1, t1 = await p1.locator(".quote").count(), time.perf_counter() - t0
await p1.close()

# --- Уровень 2: ожидание появления элемента
p2 = await context.new_page()
t0 = time.perf_counter()
await p2.goto(JS_URL, wait_until="commit")
await p2.wait_for_selector(".quote")
n2, t2 = await p2.locator(".quote").count(), time.perf_counter() - t0
await p2.close()

print(f"time.sleep(3)          : {n1} элементов за {t1:.2f} c  — всегда ждём максимум")
print(f"wait_for_selector      : {n2} элементов за {t2:.2f} c  — ждём ровно столько, сколько нужно")
print(f"Экономия на одной странице: {t1 - t2:.2f} c; на 1000 страницах: ~{(t1-t2)*1000/60:.0f} мин")

time.sleep(3)          : 10 элементов за 3.15 c  — всегда ждём максимум
wait_for_selector      : 10 элементов за 0.35 c  — ждём ровно столько, сколько нужно
Экономия на одной странице: 2.80 c; на 1000 страницах: ~47 мин


In [ ]:
# --- Уровень 3: ожидание конкретного сетевого ответа (эталон)
p3 = await context.new_page()
async with p3.expect_response(lambda r: "/api/quotes" in r.url and r.status == 200) as caught:
    await p3.goto(SCROLL_URL)
resp = await caught.value
payload = await resp.json()
print("Перехвачен ответ :", resp.url)
print("Код              :", resp.status, "| ключи:", list(payload))
print("Записей в порции :", len(payload.get("quotes", [])))
print("Есть следующая   :", payload.get("has_next"))
await p3.close()

print("""
Почему это надёжнее:
  • ждём именно те данные, которые нужны, а не «тишину в сети»;
  • сразу получаем структурированный JSON вместо парсинга DOM;
  • узнаём адрес API — дальше браузер можно выключить (блок 6).""")

Перехвачен ответ : https://quotes.toscrape.com/api/quotes?page=1
Код              : 200 | ключи: ['has_next', 'page', 'quotes', 'tag', 'top_ten_tags']
Записей в порции : 10
Есть следующая   : True

Почему это надёжнее:
  • ждём именно те данные, которые нужны, а не «тишину в сети»;
  • сразу получаем структурированный JSON вместо парсинга DOM;
  • узнаём адрес API — дальше браузер можно выключить (блок 6).


### Обработка таймаута: ошибка должна быть диагностируемой

При сбое сохраняем скриншот и HTML. Без артефактов отладка сводится к угадыванию.

In [ ]:
os.makedirs("artifacts", exist_ok=True)

p4 = await context.new_page()
await p4.goto(JS_URL, wait_until="domcontentloaded")
try:
    await p4.wait_for_selector("#element-kotorogo-net", timeout=2500)
except PWTimeout as e:
    await p4.screenshot(path="artifacts/fail_demo.png", full_page=False)
    with open("artifacts/fail_demo.html", "w", encoding="utf-8") as f:
        f.write(await p4.content())
    print("Поймали TimeoutError — это ожидаемое поведение, а не сбой скрипта.")
    print("Сохранены артефакты:", os.listdir("artifacts"))
    print("Первая строка сообщения:", str(e).splitlines()[0][:120])
await p4.close()

Поймали TimeoutError — это ожидаемое поведение, а не сбой скрипта.
Сохранены артефакты: ['fail_demo.html', 'fail_demo.png']
Первая строка сообщения: Page.wait_for_selector: Timeout 2500ms exceeded.


## 5. Бесконечная прокрутка

Страница `/scroll` подгружает цитаты по мере прокрутки. Реализуем цикл с тремя предохранителями: максимум итераций, остановка при отсутствии прироста, лимит записей.

In [ ]:
async def scroll_collect(page, url, item_selector=".quote", max_rounds=15,
                         max_items=100, stall_limit=2, pause=0.6):
    """Сбор с бесконечной прокруткой. Возвращает (записи, журнал прокрутки)."""
    await page.goto(url, wait_until="domcontentloaded")
    await page.wait_for_selector(item_selector)
    log, prev, stalls = [], 0, 0

    for r in range(max_rounds):
        await page.mouse.wheel(0, 20_000)                # либо keyboard.press("End")
        try:                                             # ждём прирост числа элементов
            await page.wait_for_function(
                "([sel, n]) => document.querySelectorAll(sel).length > n",
                arg=[item_selector, prev], timeout=4000)
        except PWTimeout:
            pass
        cur = await page.locator(item_selector).count()
        log.append({"итерация": r + 1, "элементов": cur, "прирост": cur - prev})
        if cur == prev:
            stalls += 1
            if stalls >= stall_limit:                     # предохранитель 1
                break
        else:
            stalls = 0
        prev = cur
        if cur >= max_items:                             # предохранитель 2
            break
        await asyncio.sleep(pause)                       # вежливость к серверу

    items = await page.eval_on_selector_all(item_selector, """
        els => els.map(el => ({
            text:   el.querySelector('.text')?.innerText?.trim() ?? null,
            author: el.querySelector('.author')?.innerText?.trim() ?? null
        }))""")
    return items[:max_items], pd.DataFrame(log)

t0 = time.perf_counter()
items, scroll_log = await scroll_collect(page, SCROLL_URL, max_items=60)
browser_time = time.perf_counter() - t0

print(f"Собрано {len(items)} записей за {browser_time:.1f} c браузером")
display(scroll_log)

Собрано 60 записей за 1.2 c браузером


,итерация,элементов,прирост
0,1,10,10
1,2,60,50


## 6. Перехват XHR → переход на прямые HTTP-запросы

Ключевой приём занятия. Браузер использован один раз — чтобы узнать адрес и формат эндпоинта. Дальше сбор идёт обычными запросами: быстрее, стабильнее, дешевле.

In [ ]:
# 6.1 Каталогизируем XHR-запросы страницы
p5 = await context.new_page()
seen = []
p5.on("response", lambda r: seen.append((r.status, r.request.resource_type, r.url)))
await p5.goto(SCROLL_URL, wait_until="load")
await p5.mouse.wheel(0, 20_000)
await p5.wait_for_timeout(1500)
await p5.close()

api_calls = [s for s in seen if s[1] in ("xhr", "fetch")]
print("Всего ответов:", len(seen), "| из них XHR/fetch:", len(api_calls))
for st, rt, url in api_calls[:6]:
    print(f"  {st} {rt:6} {url}")

Всего ответов: 8 | из них XHR/fetch: 2
  200 xhr    https://quotes.toscrape.com/api/quotes?page=1
  200 xhr    https://quotes.toscrape.com/api/quotes?page=2


In [ ]:
# 6.2 Тот же сбор без браузера
def collect_via_api(max_items=60, pause=0.3):
    s = requests.Session()
    s.headers["User-Agent"] = "course-crawler/1.0 (educational use)"
    out, page_no = [], 1
    while len(out) < max_items and page_no <= 20:          # предохранитель
        r = s.get("https://quotes.toscrape.com/api/quotes",
                  params={"page": page_no}, timeout=15)
        r.raise_for_status()
        body = r.json()
        batch = body.get("quotes", [])
        if not batch:
            break
        out.extend(batch)
        if not body.get("has_next"):
            break
        page_no += 1
        time.sleep(pause)
    return out[:max_items], page_no

t0 = time.perf_counter()
api_items, pages = collect_via_api(max_items=60)
api_time = time.perf_counter() - t0

print(f"Браузер (прокрутка): {len(items):>3} записей за {browser_time:5.1f} c")
print(f"Прямой API         : {len(api_items):>3} записей за {api_time:5.1f} c (страниц: {pages})")
print(f"Ускорение          : ×{browser_time/max(api_time, 1e-9):.1f}")
print("\nПамять: браузер ~200-400 МБ на экземпляр против ~30 МБ у HTTP-клиента.")

df_api = pd.json_normalize(api_items, sep="_")
print("\nПоля, полученные из API (в DOM их части не было):")
print(list(df_api.columns))
display(df_api[["author_name", "author_goodreads_link", "tags"]].head(3))

Браузер (прокрутка):  60 записей за   1.2 c
Прямой API         :  60 записей за   2.3 c (страниц: 7)
Ускорение          : ×0.5

Память: браузер ~200-400 МБ на экземпляр против ~30 МБ у HTTP-клиента.

Поля, полученные из API (в DOM их части не было):
['tags', 'text', 'author_goodreads_link', 'author_name', 'author_slug']


,author_name,author_goodreads_link,tags
0,Albert Einstein,/author/show/9810.Albert_Einstein,"[change, deep-thoughts, thinking, world]"
1,J.K. Rowling,/author/show/1077326.J_K_Rowling,"[abilities, choices]"
2,Albert Einstein,/author/show/9810.Albert_Einstein,"[inspirational, life, live, miracle, miracles]"


**Вывод для студентов:** браузер — инструмент разведки и последнего средства. В отчёте по лабораторной работе требуется явно доказать, что задача не решается через найденный XHR-эндпоинт или JSON в разметке.

## 7. Авторизация и переиспользование сессии

Логинимся один раз, сохраняем `storage_state` в файл и в дальнейших запусках создаём контекст уже авторизованным. Так сокращается число обращений к форме входа и снижается риск блокировки.

Правовая оговорка: демонстрация выполняется на тренировочном сайте. Сбор данных из-под логина на реальных сервисах обычно прямо запрещён условиями использования — в лабораторной работе это допускается только на учебных площадках.

In [ ]:
STATE_FILE = "auth_state.json"

login_ctx = await browser.new_context()
lp = await login_ctx.new_page()
await lp.goto(LOGIN_URL)

await lp.get_by_label("Username").fill("student")        # локаторы по метке формы
await lp.get_by_label("Password").fill("any-password")   # на этом сайте пароль любой
await lp.get_by_role("button", name="Login").click()     # клик с автоожиданием
await lp.wait_for_load_state("domcontentloaded")

logged_in = await lp.get_by_role("link", name="Logout").count() > 0
print("Вход выполнен:", logged_in)
print("CSRF-токен в форме был обязательным полем — при сборе через requests его нужно извлекать вручную.")

await login_ctx.storage_state(path=STATE_FILE)
print("Состояние сохранено:", STATE_FILE, "| размер:", os.path.getsize(STATE_FILE), "байт")
await lp.close(); await login_ctx.close()

# Новый контекст сразу авторизован — повторный вход не требуется
restored = await browser.new_context(storage_state=STATE_FILE)
rp = await restored.new_page()
await rp.goto(STATIC_URL)
print("В восстановленном контексте авторизация активна:",
      await rp.get_by_role("link", name="Logout").count() > 0)
await rp.close(); await restored.close()

Вход выполнен: True
CSRF-токен в форме был обязательным полем — при сборе через requests его нужно извлекать вручную.
Состояние сохранено: auth_state.json | размер: 339 байт
В восстановленном контексте авторизация активна: True


## 8. Экономия ресурсов и экономика решения

Блокировка изображений, шрифтов, медиа и аналитики — самый простой способ ускорить браузерный сбор. Измеряем эффект на живой странице.

In [ ]:
BLOCK = {"image", "media", "font"}

async def timed_load(url, block=False):
    ctx = await browser.new_context()
    p = await ctx.new_page()
    stats = {"requests": 0, "bytes": 0, "blocked": 0}

    if block:
        async def router(route):
            if route.request.resource_type in BLOCK:
                stats["blocked"] += 1
                return await route.abort()
            return await route.continue_()
        await p.route("**/*", router)

    async def on_resp(r):
        stats["requests"] += 1
        try:
            stats["bytes"] += len(await r.body())
        except Exception:
            pass
    p.on("response", on_resp)

    t0 = time.perf_counter()
    await p.goto(url, wait_until="load")
    await p.wait_for_selector(".quote")
    stats["сек"] = round(time.perf_counter() - t0, 2)
    await asyncio.sleep(0.5)            # даём обработчикам дочитать тела ответов
    await ctx.close()
    return stats

a = await timed_load(JS_URL, block=False)
b = await timed_load(JS_URL, block=True)
display(pd.DataFrame([{"режим": "всё загружаем", **a}, {"режим": "блокируем медиа", **b}]))
print("На сайтах с фотогалереями выигрыш достигает 3-5× по времени и 10× по трафику.")

,режим,requests,bytes,blocked,сек
0,всё загружаем,6,141871,0,0.63
1,блокируем медиа,5,94975,1,0.67


На сайтах с фотогалереями выигрыш достигает 3-5× по времени и 10× по трафику.


In [ ]:
# Сводная экономика: стоимость одной записи разными способами
cost = pd.DataFrame([
    {"способ": "JSON из <script> (requests)", "записей": len(df_script),
     "секунд": round(http_time, 3), "инфраструктура": "HTTP-клиент, ~30 МБ RAM"},
    {"способ": "XHR-эндпоинт (requests)", "записей": len(api_items),
     "секунд": round(api_time, 2), "инфраструктура": "HTTP-клиент, ~30 МБ RAM"},
    {"способ": "Браузер + прокрутка", "записей": len(items),
     "секунд": round(browser_time, 2), "инфраструктура": "Chromium, ~300 МБ RAM"},
])
cost["сек/запись"] = (cost["секунд"] / cost["записей"]).round(4)
display(cost)

,способ,записей,секунд,инфраструктура,сек/запись
0,JSON из <script> (requests),10,0.00,"HTTP-клиент, ~30 МБ RAM",0.0000
1,XHR-эндпоинт (requests),60,2.32,"HTTP-клиент, ~30 МБ RAM",0.0387
2,Браузер + прокрутка,60,1.16,"Chromium, ~300 МБ RAM",0.0193


### No-code эквивалент того же сценария

**Octoparse** (визуальный браузер внутри инструмента):
1. New Task → вставить URL `https://quotes.toscrape.com/scroll`.
2. Auto-detect web page data — инструмент сам предлагает поля (текст, автор).
3. В workflow добавить шаг **Scroll Page**: режим «до конца страницы», 10–15 повторов, интервал 2 с — это визуальный аналог нашего цикла из блока 5.
4. При необходимости — шаг **Click Item** в цикле для кнопки «Загрузить ещё» и **Wait before action** вместо ожиданий Playwright.
5. Run → Cloud (расписание, ротация IP) → экспорт в CSV / Google Sheets.
Логин настраивается шагами Enter Text + Click; сессия сохраняется в задаче — концептуально это то же `storage_state`.

**n8n** (собственного браузера нет, рендеринг делегируется):
1. `Schedule Trigger` — расписание.
2. `HTTP Request` к рендеринг-API (ScrapingBee / ScraperAPI / Browserless) с параметрами `render_js=true`, `wait_for=.quote` — возвращается готовый HTML.
3. `HTML Extract` — CSS-селекторы `.quote .text`, `.quote .author`, режим «вернуть массив».
4. `Split Out` → `Set`/`Code` — нормализация полей.
5. `Loop Over Items` + `Wait` — батчи и контроль скорости; ветка ошибки узла → уведомление.
6. `Postgres` / `Google Sheets` — выгрузка.
Важная оговорка для студентов: запросы с JS-рендерингом тарифицируются кратно дороже обычных, поэтому даже в no-code сценарии выгодно сначала проверить наличие XHR-эндпоинта.


## 9. Завершение работы

Браузер, контексты и Playwright нужно закрывать явно — иначе процессы `chromium` остаются в памяти. В скриптах используйте менеджер контекста `with sync_playwright() as pw:`.

In [ ]:
await context.close()
await browser.close()
await pw.stop()
print("Ресурсы освобождены.")
print("В .py-скриптах предпочтительно:  with sync_playwright() as pw: ...  — закрытие гарантировано.")

Ресурсы освобождены.
В .py-скриптах предпочтительно:  with sync_playwright() as pw: ...  — закрытие гарантировано.


## Итоги и переход к лабораторной работе

Что студент должен унести с занятия:

1. Алгоритм диагностики: View Source → поиск текста → Network/XHR → только потом браузер.
2. Умение вытащить JSON из `<script>` и из перехваченного XHR.
3. Шаблон прокрутки с тремя предохранителями (итерации, отсутствие прироста, лимит записей).
4. Три уровня ожиданий и понимание, почему `time.sleep` — антипаттерн.
5. Приоритет локаторов и запрет структурного XPath.
6. `storage_state` для переиспользования авторизации, `page.route` для блокировки медиа.
7. Сохранение скриншота и HTML при сбое.
8. Способность обосновать выбор между кодом, Octoparse и n8n по измеренным метрикам.

**Лабораторная работа (6 часов):** собрать ≥ 200 записей с динамического источника браузерным способом, найти и использовать XHR-эндпоинт, измерить и сравнить оба подхода, повторить сценарий в no-code и оформить отчёт. Полное условие — в отдельном файле задания.

### Типичные ошибки студентов (предупредить заранее)

| Ошибка | Последствие | Что требовать |
|---|---|---|
| Сразу запускают браузер | сбор медленнее в 10–50 раз | протокол диагностики в отчёте |
| `time.sleep` вместо ожиданий | медленно и всё равно нестабильно | `wait_for_selector` / `expect_response` |
| Прокрутка `while True` | бесконечный цикл, зависший ноутбук | три предохранителя |
| Структурный XPath и классы вида `.css-1x7f2kq` | поломка после ближайшего релиза сайта | локаторы по роли/тексту/data-атрибутам |
| Не закрывают браузер | утечка памяти, десятки процессов | `with` или явный `close()` |
| Логин при каждом запуске | лишняя нагрузка, риск блокировки | `storage_state` |
| Нет артефактов при сбое | невозможно диагностировать | скриншот + HTML в `except` |
| Загружают картинки и шрифты | лишний трафик и время | `page.route` с `abort()` |